# PatchCore — Multi-Anatomy MRI Anomaly Detection

This notebook runs **three PatchCore models** for artifact detection:

| # | Anatomy | Dataset | Description |
|---|---------|---------|-------------|
| 1 | **Knee** | FastMRI (preprocessed) | Single anatomy, single contrast |
| 2 | **Brain** | IXI (balanced T1 + T2) | Single anatomy, balanced contrast |
| 3 | **Combined** | Both datasets merged | Multi-anatomy generalist model |

## Method (Roth et al., CVPR 2022)
PatchCore is a **memory-bank** anomaly detector — no gradient-based training:
1. Extract **patch-level features** from a pretrained backbone (layers 2 and 3)
2. Build a **coreset memory bank** from training features (greedy subsampling)
3. Score test images by **max patch-level kNN distance** to the memory bank

## Architecture
- **Wide ResNet-50-2** pretrained on ImageNet (standard PatchCore backbone)
- Grayscale to 3-channel by repeating (for pretrained weights compatibility)
- **Multi-scale features**: layer2 (512-d) + layer3 (1024-d) = 1536-d patch features
- **Spatial pooling**: 24x24 to 6x6 = 36 patches per image (memory-efficient)
- **Coreset**: 1% greedy subsampling (paper default)
- **Balanced brain sampling** — same deterministic subset as MAE/DINO/SimCLR/DAE

## Memory Budget (Kaggle ~13 GB RAM)
| Component | Knee (12.8K) | Combined (25.7K) |
|-----------|-------------|------------------|
| Pooled patches (6x6) | 463K x 1536 = 2.8 GB | 926K x 1536 = 5.7 GB |
| After 100K cap | 614 MB | 614 MB |
| After 1% coreset | 6 MB | 6 MB |
| Global features | 105 MB | 211 MB |
| **Peak RAM** | **~3.5 GB** | **~6.5 GB** |

## Dual Anomaly Scoring
| Method | Description |
|--------|-------------|
| **PatchCore (patch kNN)** | Max patch-level kNN distance — native PatchCore scoring |
| **Global kNN** | Cosine kNN on GAP features — same as SimCLR/MAE/DINO/DAE |

## Consistency with Other Methods
| Setting | PatchCore | SimCLR | MAE | DINO | DAE |
|---------|-----------|--------|-----|------|-----|
| Image size | 192x192 | 192x192 | 192x192 | 192x192 | 192x192 |
| Brain sampling | Balanced (same) | Balanced (same) | Balanced (same) | Balanced (same) | Balanced (same) |
| Evaluation | kNN + patch + collapse + t-SNE | kNN + collapse + t-SNE | kNN + recon + collapse + t-SNE | kNN + collapse + t-SNE | kNN + recon + collapse + t-SNE |

## Time Budget
PatchCore has **no training loop** — only feature extraction + coreset construction.
Expected total: **~1-2 hours** (much faster than gradient-based methods).

In [ ]:
import os
import time
import gc
import numpy as np
import random
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from torchvision.models import wide_resnet50_2, Wide_ResNet50_2_Weights

from tqdm import tqdm
from sklearn.neighbors import NearestNeighbors
from sklearn.manifold import TSNE

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
if device == "cuda":
    print(torch.cuda.get_device_name(0))
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Hyperparameters
BATCH_SIZE     = 32
IMG_SIZE       = 192
NUM_WORKERS    = 4

# PatchCore-specific
PATCH_POOL_SIZE = 6           # pool spatial grid to 6x6 = 36 patches/image
MAX_MEMORY_PATCHES = 100_000  # cap patches before coreset (RAM safety)
CORESET_RATIO   = 0.01        # 1% — paper default (Roth et al., CVPR 2022)
PATCH_K         = 3           # k for patch-level kNN scoring

# Evaluation (same as all other notebooks)
K_NEIGHBORS    = 5

# Memory estimates
patches_per_image = PATCH_POOL_SIZE ** 2
patch_dim = 512 + 1024  # 1536
bytes_per_patch = patch_dim * 4
print(f"Image size: {IMG_SIZE}")
print(f"Patches per image: {patches_per_image} ({PATCH_POOL_SIZE}x{PATCH_POOL_SIZE})")
print(f"Patch dim: {patch_dim}")
print(f"Max memory patches: {MAX_MEMORY_PATCHES:,}")
print(f"Coreset ratio: {CORESET_RATIO} (paper default)")
print(f"\nMemory per 10K images: {10000 * patches_per_image * bytes_per_patch / 1e9:.2f} GB")
print(f"After 100K cap: {MAX_MEMORY_PATCHES * bytes_per_patch / 1e6:.0f} MB")
print(f"After 1% coreset: {int(MAX_MEMORY_PATCHES * CORESET_RATIO) * bytes_per_patch / 1e6:.1f} MB")

In [ ]:
# Balanced Sampling + Dataset

def get_balanced_paths(root_dir, target_total=None):
    """Deterministic balanced subsampling — IDENTICAL to MAE/DINO/SimCLR/DAE notebooks."""
    groups = {}
    for dirpath, _, filenames in os.walk(root_dir):
        npy_files = sorted(f for f in filenames if f.endswith('.npy'))
        if npy_files:
            folder = os.path.basename(dirpath)
            groups[folder] = [os.path.join(dirpath, f) for f in npy_files]

    if not groups:
        return [], []

    if target_total is None:
        paths, labels = [], []
        for folder in sorted(groups):
            paths.extend(groups[folder])
            labels.extend([folder] * len(groups[folder]))
        return paths, labels

    n_groups = len(groups)
    target_per_group = target_total // n_groups

    paths, labels = [], []
    for folder in sorted(groups):
        group_paths = groups[folder]
        if len(group_paths) <= target_per_group:
            selected = group_paths
        else:
            idx = np.round(np.linspace(0, len(group_paths) - 1, target_per_group)).astype(int)
            selected = [group_paths[i] for i in idx]
        paths.extend(selected)
        labels.extend([folder] * len(selected))
        print(f"    {folder}: {len(selected)}/{len(group_paths)} selected")

    return paths, labels


class MRIDataset(Dataset):
    def __init__(self, paths, labels, label_map, name=""):
        self.paths = paths
        self.label_map = label_map
        self.labels = [label_map[l] for l in labels]
        print(f"  [{name}] {len(paths)} slices | Labels: {label_map}")

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = np.load(self.paths[idx]).astype(np.float32)
        img = np.clip(img, 0.0, 1.0)
        return torch.from_numpy(img).unsqueeze(0).float(), self.labels[idx]

In [ ]:
# PatchCore Feature Extractor

class PatchCoreExtractor(nn.Module):
    """Wide ResNet-50-2 feature extractor for PatchCore.
    Extracts patch-level features from layers 2 & 3,
    and global features (GAP) for kNN evaluation."""

    def __init__(self):
        super().__init__()
        backbone = wide_resnet50_2(weights=Wide_ResNet50_2_Weights.IMAGENET1K_V1)
        self.layer0 = nn.Sequential(
            backbone.conv1, backbone.bn1, backbone.relu, backbone.maxpool,
        )
        self.layer1 = backbone.layer1
        self.layer2 = backbone.layer2  # 512-d, H/8
        self.layer3 = backbone.layer3  # 1024-d, H/16
        self.layer4 = backbone.layer4  # 2048-d, H/32
        self.avgpool = backbone.avgpool

        for p in self.parameters():
            p.requires_grad = False

        self.patch_dim = 512 + 1024  # 1536
        self.global_dim = 2048

    def forward(self, x, pool_size=None):
        """x: (B, 1, H, W) grayscale.
        Returns:
            patch_features: (B, P, 1536)
            global_features: (B, 2048)
        """
        x = x.repeat(1, 3, 1, 1)

        mean = torch.tensor([0.485, 0.456, 0.406], device=x.device).view(1, 3, 1, 1)
        std = torch.tensor([0.229, 0.224, 0.225], device=x.device).view(1, 3, 1, 1)
        x = (x - mean) / std

        x = self.layer0(x)
        x = self.layer1(x)
        f2 = self.layer2(x)    # (B, 512, 24, 24)
        f3 = self.layer3(f2)   # (B, 1024, 12, 12)
        f4 = self.layer4(f3)   # (B, 2048, 6, 6)

        global_feat = self.avgpool(f4).flatten(1)  # (B, 2048)

        target_h, target_w = f2.shape[2], f2.shape[3]
        f3_up = F.interpolate(f3, size=(target_h, target_w), mode='bilinear', align_corners=False)

        patch_feat = torch.cat([f2, f3_up], dim=1)  # (B, 1536, 24, 24)

        if pool_size is not None:
            patch_feat = F.adaptive_avg_pool2d(patch_feat, (pool_size, pool_size))

        B, D, H, W = patch_feat.shape
        patch_feat = patch_feat.permute(0, 2, 3, 1).reshape(B, H * W, D)

        return patch_feat, global_feat


_ext = PatchCoreExtractor()
print(f"Backbone params: {sum(p.numel() for p in _ext.parameters()):,} "
      f"({sum(p.numel() for p in _ext.parameters())/1e6:.1f}M) — all frozen")
print(f"Patch feature dim: {_ext.patch_dim}")
print(f"Global feature dim: {_ext.global_dim}")

with torch.no_grad():
    _inp = torch.randn(2, 1, 192, 192)
    _pf, _gf = _ext(_inp, pool_size=PATCH_POOL_SIZE)
    print(f"Input: {_inp.shape}")
    print(f"Patch features: {_pf.shape} -> {_pf.shape[1]} patches/image, {_pf.shape[2]}-d")
    print(f"Global features: {_gf.shape}")
del _ext, _inp, _pf, _gf

In [ ]:
# PatchCore Core Functions (Memory-Efficient)
TOTAL_TIME_BUDGET = 12 * 3600
SESSION_START = time.time()


def log_time_budget(phase=""):
    elapsed = time.time() - SESSION_START
    remaining = TOTAL_TIME_BUDGET - elapsed
    print(f"  [{phase}] Elapsed: {elapsed/3600:.2f}h | Remaining: {remaining/3600:.2f}h "
          f"| Budget used: {100*elapsed/TOTAL_TIME_BUDGET:.1f}%")
    if remaining < 1800:
        print("  WARNING: Less than 30 min remaining!")
    return remaining


def log_gpu_memory(prefix=""):
    if device == "cuda":
        alloc = torch.cuda.memory_allocated() / 1e9
        peak = torch.cuda.max_memory_allocated() / 1e9
        total = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f"  GPU [{prefix}]: {alloc:.2f}/{total:.1f} GB (peak: {peak:.2f} GB)")


def clear_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
    log_gpu_memory("after clear")
    log_time_budget("GPU cleared")


@torch.no_grad()
def extract_global_features(extractor, loader):
    """Extract L2-normalized global features (2048-d) from all images.
    Memory: N x 2048 x 4 bytes (~105 MB for 12K images). Safe."""
    extractor.eval()
    all_feats, all_labels = [], []
    for imgs, lbls in tqdm(loader, desc="Global features"):
        imgs = imgs.to(device)
        with torch.amp.autocast("cuda", enabled=(device == "cuda")):
            _, gf = extractor(imgs, pool_size=None)
        gf = F.normalize(gf.float(), dim=1)
        all_feats.append(gf.cpu().numpy())
        all_labels.append(lbls.numpy())
    return np.concatenate(all_feats), np.concatenate(all_labels)


@torch.no_grad()
def build_memory_bank_streaming(extractor, loader, pool_size, max_patches, seed=42):
    """Build patch feature pool by streaming through the dataset.
    Uses reservoir sampling to cap at max_patches without holding everything in RAM.

    Returns: (capped_patches, n_total_seen)
    """
    extractor.eval()
    rng = np.random.RandomState(seed)

    reservoir = None
    n_seen = 0
    D = None

    for imgs, _ in tqdm(loader, desc="Building patch pool"):
        imgs = imgs.to(device)
        with torch.amp.autocast("cuda", enabled=(device == "cuda")):
            pf, _ = extractor(imgs, pool_size=pool_size)
        pf = pf.float().cpu().numpy()  # (B, P, D)
        B, P, D_batch = pf.shape

        if D is None:
            D = D_batch
            reservoir = np.empty((max_patches, D), dtype=np.float32)

        patches_flat = pf.reshape(B * P, D)
        n_new = len(patches_flat)

        for j in range(n_new):
            if n_seen < max_patches:
                reservoir[n_seen] = patches_flat[j]
            else:
                k = rng.randint(0, n_seen + 1)
                if k < max_patches:
                    reservoir[k] = patches_flat[j]
            n_seen += 1

    actual = min(n_seen, max_patches)
    print(f"  Total patches seen: {n_seen:,} | Kept: {actual:,} "
          f"({actual * D * 4 / 1e6:.0f} MB)")
    return reservoir[:actual], n_seen


def greedy_coreset(features, ratio=0.01, seed=42):
    """Greedy coreset selection (furthest-point sampling).
    Paper default: 1% ratio."""
    n_total = len(features)
    n_select = max(1, int(n_total * ratio))
    print(f"  Coreset: selecting {n_select} from {n_total} ({100*ratio:.1f}%)")

    if n_select >= n_total:
        return features, np.arange(n_total)

    rng = np.random.RandomState(seed)

    norms = np.linalg.norm(features, axis=1, keepdims=True)
    norms = np.maximum(norms, 1e-8)
    feat_norm = features / norms

    selected = [rng.randint(0, n_total)]
    min_dist = np.full(n_total, np.inf, dtype=np.float32)

    for i in tqdm(range(1, n_select), desc="  Coreset selection", leave=False):
        last = feat_norm[selected[-1]]
        dist_to_last = 1.0 - feat_norm @ last  # cosine distance
        min_dist = np.minimum(min_dist, dist_to_last)
        next_idx = np.argmax(min_dist)
        selected.append(next_idx)

    selected = np.array(selected)
    print(f"  Coreset done: {len(selected)} vectors, "
          f"{len(selected) * features.shape[1] * 4 / 1e6:.1f} MB")
    return features[selected], selected


@torch.no_grad()
def score_images_streaming(extractor, loader, memory_bank, pool_size, k=3):
    """Score images in streaming fashion — never holds all val patches in RAM.
    For each batch: extract patches, kNN against memory bank, aggregate.
    Returns: (n_images,) image-level anomaly scores."""
    extractor.eval()

    nn_model = NearestNeighbors(n_neighbors=k, metric="cosine", algorithm="auto")
    nn_model.fit(memory_bank)
    print(f"  Memory bank: {memory_bank.shape} | kNN k={k}")

    all_scores = []
    for imgs, _ in tqdm(loader, desc="Patch scoring"):
        imgs = imgs.to(device)
        with torch.amp.autocast("cuda", enabled=(device == "cuda")):
            pf, _ = extractor(imgs, pool_size=pool_size)
        pf = pf.float().cpu().numpy()  # (B, P, D)
        B, P, D = pf.shape

        pf_flat = pf.reshape(B * P, D)
        dists, _ = nn_model.kneighbors(pf_flat)
        patch_dists = dists.mean(axis=1).reshape(B, P)

        image_scores = patch_dists.max(axis=1)
        all_scores.append(image_scores)

    return np.concatenate(all_scores)

In [ ]:
# Full PatchCore Pipeline

def run_patchcore(train_loader, val_loader, val_dataset,
                  extractor, save_dir, model_name="PatchCore"):
    os.makedirs(save_dir, exist_ok=True)

    remaining = log_time_budget(f"START {model_name}")

    print(f"\n{'='*60}")
    print(f"  PatchCore: {model_name}")
    print(f"  Save dir: {save_dir}")
    print(f"  Pool: {PATCH_POOL_SIZE}x{PATCH_POOL_SIZE}, "
          f"max_patches: {MAX_MEMORY_PATCHES:,}, coreset: {CORESET_RATIO}")
    print(f"{'='*60}\n")

    # 1. Build Patch Memory Bank (streaming + capped)
    print("[1/5] Building patch pool (streaming reservoir sampling)...")
    t0 = time.time()
    patch_pool, n_total_patches = build_memory_bank_streaming(
        extractor, train_loader, pool_size=PATCH_POOL_SIZE,
        max_patches=MAX_MEMORY_PATCHES, seed=SEED)
    print(f"  Patch pool: {patch_pool.shape} ({time.time()-t0:.1f}s)")

    # 2. Coreset Selection
    print("\n[2/5] Greedy coreset selection...")
    t0 = time.time()
    memory_bank, coreset_idx = greedy_coreset(
        patch_pool, ratio=CORESET_RATIO, seed=SEED)
    print(f"  Memory bank: {memory_bank.shape} ({time.time()-t0:.1f}s)")

    del patch_pool
    gc.collect()

    np.save(os.path.join(save_dir, "memory_bank.npy"), memory_bank)
    print(f"  memory_bank.npy: {memory_bank.nbytes / 1e6:.1f} MB")

    # 3. Extract Global Features
    print("\n[3/5] Extracting global features (for kNN evaluation)...")
    t0 = time.time()
    train_global, train_labels = extract_global_features(extractor, train_loader)
    val_global, val_labels = extract_global_features(extractor, val_loader)
    print(f"  Train: {train_global.shape} | Val: {val_global.shape} ({time.time()-t0:.1f}s)")

    # 4. PatchCore Scoring (streaming)
    print("\n[4/5] PatchCore scoring (streaming batch-by-batch)...")
    t0 = time.time()
    print("  Scoring train images...")
    train_patch_scores = score_images_streaming(
        extractor, train_loader, memory_bank, PATCH_POOL_SIZE, k=PATCH_K)
    print("  Scoring val images...")
    val_patch_scores = score_images_streaming(
        extractor, val_loader, memory_bank, PATCH_POOL_SIZE, k=PATCH_K)
    print(f"  Done ({time.time()-t0:.1f}s)")

    # 5. Global kNN Scoring
    print("\n[5/5] Global kNN scoring...")
    nn_global = NearestNeighbors(n_neighbors=K_NEIGHBORS, metric="cosine")
    nn_global.fit(train_global)
    train_knn = nn_global.kneighbors(train_global)[0].mean(axis=1)
    val_knn = nn_global.kneighbors(val_global)[0].mean(axis=1)

    log_time_budget(f"PIPELINE DONE {model_name}")

    return {
        "memory_bank": memory_bank,
        "train_global": train_global,
        "val_global": val_global,
        "train_labels": train_labels,
        "val_labels": val_labels,
        "train_patch_scores": train_patch_scores,
        "val_patch_scores": val_patch_scores,
        "train_knn": train_knn,
        "val_knn": val_knn,
    }

In [ ]:
# Full Analysis Pipeline

def run_full_analysis(results, val_dataset, save_dir, model_name):
    analysis_start = time.time()
    log_time_budget(f"ANALYSIS START {model_name}")

    train_global = results["train_global"]
    val_global = results["val_global"]
    train_labels = results["train_labels"]
    val_labels = results["val_labels"]
    train_patch_scores = results["train_patch_scores"]
    val_patch_scores = results["val_patch_scores"]
    train_knn = results["train_knn"]
    val_knn = results["val_knn"]

    print(f"\n{'='*60}")
    print(f"  Analysis: {model_name}")
    print(f"{'='*60}")

    # 1. Collapse Check
    print("\n[1/8] Collapse check...")
    n_check = min(2000, len(train_global))
    for name, feats in [("Train", train_global[:n_check]), ("Val", val_global[:n_check])]:
        std_val = np.std(feats, axis=0).mean()
        sim_mat = np.dot(feats, feats.T)
        off_diag = sim_mat[np.triu_indices_from(sim_mat, k=1)]
        mean_sim = off_diag.mean()
        s_std = "GOOD" if std_val > 0.01 else "COLLAPSE"
        s_sim = "GOOD" if mean_sim < 0.9 else "HIGH"
        print(f"  {name}: STD={std_val:.4f} [{s_std}] | Mean cos sim={mean_sim:.4f} [{s_sim}]")

    # 2. PatchCore Anomaly Scores
    print("\n[2/8] PatchCore anomaly scores...")
    print(f"  Train patch — mean: {train_patch_scores.mean():.4f}, std: {train_patch_scores.std():.4f}")
    print(f"  Val patch   — mean: {val_patch_scores.mean():.4f}, std: {val_patch_scores.std():.4f}")

    # 3. Global kNN Anomaly Scores
    print("\n[3/8] Global kNN anomaly scores...")
    print(f"  Train kNN — mean: {train_knn.mean():.4f}, std: {train_knn.std():.4f}")
    print(f"  Val kNN   — mean: {val_knn.mean():.4f}, std: {val_knn.std():.4f}")

    # 4. Score Distributions
    print("\n[4/8] Score distributions...")
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle(f"{model_name} — Anomaly Score Distributions", fontsize=14, y=1.02)
    axes[0][0].hist(train_patch_scores, bins=60, alpha=0.7, color="steelblue", edgecolor="white")
    axes[0][0].set_title("Train — PatchCore Score"); axes[0][0].set_xlabel("Max Patch Dist")
    axes[0][1].hist(val_patch_scores, bins=60, alpha=0.7, color="coral", edgecolor="white")
    axes[0][1].set_title("Val — PatchCore Score"); axes[0][1].set_xlabel("Max Patch Dist")
    axes[1][0].hist(train_knn, bins=60, alpha=0.7, color="steelblue", edgecolor="white")
    axes[1][0].set_title("Train — Global kNN"); axes[1][0].set_xlabel("Cosine Dist")
    axes[1][1].hist(val_knn, bins=60, alpha=0.7, color="coral", edgecolor="white")
    axes[1][1].set_title("Val — Global kNN"); axes[1][1].set_xlabel("Cosine Dist")
    fig.tight_layout()
    fig.savefig(os.path.join(save_dir, "anomaly_scores.png"), dpi=150, bbox_inches="tight")
    plt.show()

    # 5. Threshold Analysis
    print("[5/8] Threshold analysis...")
    print("  PatchCore-based:")
    for pct in [90, 95, 97.5, 99]:
        thresh = np.percentile(train_patch_scores, pct)
        flagged = (val_patch_scores > thresh).sum()
        print(f"    p{pct}: thresh={thresh:.4f} -> {flagged}/{len(val_patch_scores)} val flagged "
              f"({100*flagged/len(val_patch_scores):.1f}%)")
    print("  kNN-based:")
    for pct in [90, 95, 97.5, 99]:
        thresh = np.percentile(train_knn, pct)
        flagged = (val_knn > thresh).sum()
        print(f"    p{pct}: thresh={thresh:.4f} -> {flagged}/{len(val_knn)} val flagged "
              f"({100*flagged/len(val_knn):.1f}%)")

    # 6. t-SNE
    print("[6/8] t-SNE...")
    tsne_start = time.time()
    N_TSNE = min(3000, len(val_global))
    tsne = TSNE(n_components=2, perplexity=30, random_state=SEED, max_iter=1000)
    emb = tsne.fit_transform(val_global[:N_TSNE])
    labels_sub = val_labels[:N_TSNE]
    knn_sub = val_knn[:N_TSNE]
    patch_sub = val_patch_scores[:N_TSNE]
    print(f"  t-SNE done ({time.time()-tsne_start:.1f}s)")

    fig, axes = plt.subplots(1, 4, figsize=(22, 5.5))
    fig.suptitle(f"{model_name} — t-SNE Embeddings", fontsize=14, y=1.02)
    axes[0].scatter(emb[:, 0], emb[:, 1], s=3, alpha=0.5, c="steelblue")
    axes[0].set_title("Unsupervised"); axes[0].set_xticks([]); axes[0].set_yticks([])
    sc1 = axes[1].scatter(emb[:, 0], emb[:, 1], s=3, alpha=0.5, c=labels_sub, cmap="coolwarm")
    axes[1].set_title("By Label"); axes[1].set_xticks([]); axes[1].set_yticks([])
    plt.colorbar(sc1, ax=axes[1], shrink=0.8)
    sc2 = axes[2].scatter(emb[:, 0], emb[:, 1], s=3, alpha=0.5, c=knn_sub, cmap="YlOrRd")
    axes[2].set_title("Global kNN Score"); axes[2].set_xticks([]); axes[2].set_yticks([])
    plt.colorbar(sc2, ax=axes[2], shrink=0.8)
    sc3 = axes[3].scatter(emb[:, 0], emb[:, 1], s=3, alpha=0.5, c=patch_sub, cmap="YlOrRd")
    axes[3].set_title("PatchCore Score"); axes[3].set_xticks([]); axes[3].set_yticks([])
    plt.colorbar(sc3, ax=axes[3], shrink=0.8)
    fig.tight_layout()
    fig.savefig(os.path.join(save_dir, "tsne.png"), dpi=150, bbox_inches="tight")
    plt.show()

    # 7. Cosine Similarity Distribution
    print("[7/8] Cosine similarity distribution...")
    subset = torch.tensor(val_global[:2000])
    sim_matrix = torch.mm(subset, subset.T).numpy()
    upper_tri = sim_matrix[np.triu_indices_from(sim_matrix, k=1)]
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.hist(upper_tri, bins=100, color="mediumpurple", edgecolor="white", alpha=0.8)
    ax.axvline(upper_tri.mean(), color="red", ls="--", label=f"Mean: {upper_tri.mean():.3f}")
    ax.set_title(f"{model_name} — Pairwise Cosine Similarity")
    ax.set_xlabel("Cosine Similarity"); ax.legend()
    fig.tight_layout()
    fig.savefig(os.path.join(save_dir, "similarity_dist.png"), dpi=150, bbox_inches="tight")
    plt.show()

    # 8. Top Anomalies
    print("[8/8] Top anomalies...")
    fig, axes = plt.subplots(2, 5, figsize=(15, 6))
    fig.suptitle(f"{model_name} — Top-10 Anomalies (by PatchCore score)", fontsize=14)
    top_idx = np.argsort(val_patch_scores)[-10:][::-1]
    for i, idx in enumerate(top_idx):
        ax = axes[i // 5][i % 5]
        img = np.load(val_dataset.paths[idx])
        ax.imshow(img, cmap="gray", vmin=0, vmax=1)
        ax.set_title(f"Patch={val_patch_scores[idx]:.4f}\nkNN={val_knn[idx]:.4f}", fontsize=7)
        ax.axis("off")
    fig.tight_layout()
    fig.savefig(os.path.join(save_dir, "top_anomalies.png"), dpi=150, bbox_inches="tight")
    plt.show()

    # Save
    np.savez(
        os.path.join(save_dir, "scoring.npz"),
        train_patch_scores=train_patch_scores, val_patch_scores=val_patch_scores,
        train_knn=train_knn, val_knn=val_knn,
        train_feats=train_global, val_feats=val_global,
        train_labels=train_labels, val_labels=val_labels,
        threshold_patch_p95=np.percentile(train_patch_scores, 95),
        threshold_knn_p95=np.percentile(train_knn, 95),
    )

    print(f"\n  Analysis complete in {(time.time()-analysis_start)/60:.1f} min")
    print(f"  Files: {os.listdir(save_dir)}")
    log_time_budget(f"ANALYSIS END {model_name}")

In [ ]:
# Discover dataset paths
KNEE_ROOT = "/kaggle/input/preprocessed-fastmri-knee"
BRAIN_ROOT = "/kaggle/input/preprocessed-ixi-brain"

def show_tree(path, prefix="", max_depth=3, depth=0):
    if depth >= max_depth:
        return
    try:
        entries = sorted(os.listdir(path))
    except (PermissionError, FileNotFoundError) as e:
        print(f"{prefix}  {e}")
        return
    dirs = [e for e in entries if os.path.isdir(os.path.join(path, e))]
    files = [e for e in entries if os.path.isfile(os.path.join(path, e))]
    if files:
        npy = [f for f in files if f.endswith('.npy')]
        if npy:
            print(f"{prefix}  {len(npy)} .npy files (e.g., {npy[0]})")
        for f in [x for x in files if not x.endswith('.npy')][:3]:
            print(f"{prefix}  {f}")
    for d in dirs:
        print(f"{prefix}  {d}/")
        show_tree(os.path.join(path, d), prefix + "    ", max_depth, depth + 1)

print("=" * 50)
print("KNEE Dataset:")
print("=" * 50)
show_tree(KNEE_ROOT)

print(f"\n{'='*50}")
print("BRAIN Dataset:")
print("=" * 50)
show_tree(BRAIN_ROOT)

In [ ]:
# Configure Paths + Balanced Brain Sampling
KNEE_TRAIN = os.path.join(KNEE_ROOT, "train")
KNEE_VAL   = os.path.join(KNEE_ROOT, "val")
BRAIN_TRAIN = os.path.join(BRAIN_ROOT, "train")
BRAIN_VAL   = os.path.join(BRAIN_ROOT, "val")

print("Knee (all):")
knee_train_paths, knee_train_labels = get_balanced_paths(KNEE_TRAIN)
knee_val_paths, knee_val_labels = get_balanced_paths(KNEE_VAL)
print(f"  Train: {len(knee_train_paths)} | Val: {len(knee_val_paths)}")

print(f"\nBrain (balanced to match knee = {len(knee_train_paths)}):")
print("  Train:")
brain_train_paths, brain_train_labels = get_balanced_paths(
    BRAIN_TRAIN, target_total=len(knee_train_paths))
print("  Val:")
brain_val_paths, brain_val_labels = get_balanced_paths(
    BRAIN_VAL, target_total=len(knee_val_paths))
print(f"  Train: {len(brain_train_paths)} | Val: {len(brain_val_paths)}")

knee_label_map = {l: i for i, l in enumerate(sorted(set(knee_train_labels)))}
brain_label_map = {l: i for i, l in enumerate(sorted(set(brain_train_labels)))}
all_labels = sorted(set(knee_train_labels + brain_train_labels))
combined_label_map = {l: i for i, l in enumerate(all_labels)}

print(f"\nLabel maps:")
print(f"  Knee: {knee_label_map}")
print(f"  Brain: {brain_label_map}")
print(f"  Combined: {combined_label_map}")

for name, n in [("Knee", len(knee_train_paths)), ("Brain", len(brain_train_paths)),
                ("Combined", len(knee_train_paths) + len(brain_train_paths))]:
    total_patches = n * patches_per_image
    gb = total_patches * patch_dim * 4 / 1e9
    capped = min(total_patches, MAX_MEMORY_PATCHES)
    print(f"  {name}: {n} imgs x {patches_per_image} patches = {total_patches:,} "
          f"({gb:.1f} GB raw -> {capped:,} after cap -> "
          f"{int(capped * CORESET_RATIO):,} in coreset)")

In [ ]:
# Load Pretrained Backbone (shared across all 3)
print("Loading Wide ResNet-50-2 (ImageNet pretrained)...")
extractor = PatchCoreExtractor().to(device).eval()
log_gpu_memory("backbone loaded")
print("Ready.")

---
# Part 1: Knee Only (FastMRI)

In [ ]:
print("Loading Knee datasets...")

knee_train_ds = MRIDataset(knee_train_paths, knee_train_labels, knee_label_map, name="Knee-Train")
knee_val_ds = MRIDataset(knee_val_paths, knee_val_labels, knee_label_map, name="Knee-Val")

knee_train_loader = DataLoader(knee_train_ds, batch_size=BATCH_SIZE, shuffle=False,
                               num_workers=NUM_WORKERS, pin_memory=True)
knee_val_loader = DataLoader(knee_val_ds, batch_size=BATCH_SIZE, shuffle=False,
                             num_workers=NUM_WORKERS, pin_memory=True)

print(f"Train: {len(knee_train_ds)} | Val: {len(knee_val_ds)}")

In [ ]:
KNEE_SAVE = "/kaggle/working/checkpoints/patchcore_knee"
knee_results = run_patchcore(
    knee_train_loader, knee_val_loader, knee_val_ds,
    extractor, KNEE_SAVE, model_name="PatchCore-Knee")

In [ ]:
run_full_analysis(knee_results, knee_val_ds, KNEE_SAVE, "PatchCore-Knee")

In [ ]:
del knee_results, knee_train_loader, knee_val_loader
clear_gpu()

---
# Part 2: Brain Only (IXI — Balanced T1 + T2)

In [ ]:
print("Loading Brain datasets (balanced)...")

brain_train_ds = MRIDataset(brain_train_paths, brain_train_labels, brain_label_map, name="Brain-Train")
brain_val_ds = MRIDataset(brain_val_paths, brain_val_labels, brain_label_map, name="Brain-Val")

brain_train_loader = DataLoader(brain_train_ds, batch_size=BATCH_SIZE, shuffle=False,
                                num_workers=NUM_WORKERS, pin_memory=True)
brain_val_loader = DataLoader(brain_val_ds, batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=NUM_WORKERS, pin_memory=True)

print(f"Train: {len(brain_train_ds)} | Val: {len(brain_val_ds)}")

In [ ]:
BRAIN_SAVE = "/kaggle/working/checkpoints/patchcore_brain"
brain_results = run_patchcore(
    brain_train_loader, brain_val_loader, brain_val_ds,
    extractor, BRAIN_SAVE, model_name="PatchCore-Brain")

In [ ]:
run_full_analysis(brain_results, brain_val_ds, BRAIN_SAVE, "PatchCore-Brain")

In [ ]:
del brain_results, brain_train_loader, brain_val_loader
clear_gpu()

---
# Part 3: Combined (Knee + Balanced Brain)

In [ ]:
print("Loading Combined (Knee + balanced Brain) datasets...")

comb_train_paths = knee_train_paths + brain_train_paths
comb_train_labels = knee_train_labels + brain_train_labels
comb_val_paths = knee_val_paths + brain_val_paths
comb_val_labels = knee_val_labels + brain_val_labels

comb_train_ds = MRIDataset(comb_train_paths, comb_train_labels, combined_label_map, name="Combined-Train")
comb_val_ds = MRIDataset(comb_val_paths, comb_val_labels, combined_label_map, name="Combined-Val")

comb_train_loader = DataLoader(comb_train_ds, batch_size=BATCH_SIZE, shuffle=False,
                               num_workers=NUM_WORKERS, pin_memory=True)
comb_val_loader = DataLoader(comb_val_ds, batch_size=BATCH_SIZE, shuffle=False,
                             num_workers=NUM_WORKERS, pin_memory=True)

print(f"Combined train: {len(comb_train_ds)} | Val: {len(comb_val_ds)}")

In [ ]:
COMBINED_SAVE = "/kaggle/working/checkpoints/patchcore_combined"
comb_results = run_patchcore(
    comb_train_loader, comb_val_loader, comb_val_ds,
    extractor, COMBINED_SAVE, model_name="PatchCore-Combined")

In [ ]:
run_full_analysis(comb_results, comb_val_ds, COMBINED_SAVE, "PatchCore-Combined")

In [ ]:
del comb_results, comb_train_loader, comb_val_loader
del extractor
clear_gpu()

---
# Summary and Comparison

In [ ]:
# Cross-Model Comparison
print("\n" + "=" * 80)
print("  PATCHCORE MODEL COMPARISON SUMMARY")
print("=" * 80)

results_all = {}
for name, save_dir in [("Knee", KNEE_SAVE), ("Brain", BRAIN_SAVE), ("Combined", COMBINED_SAVE)]:
    data = np.load(os.path.join(save_dir, "scoring.npz"), allow_pickle=True)
    results_all[name] = {
        "train_patch": data["train_patch_scores"],
        "val_patch": data["val_patch_scores"],
        "train_knn": data["train_knn"],
        "val_knn": data["val_knn"],
        "val_feats": data["val_feats"],
        "thresh_patch": float(data["threshold_patch_p95"]),
        "thresh_knn": float(data["threshold_knn_p95"]),
    }

print(f"\n  PatchCore scoring (max patch distance):")
print(f"  {'Model':<12} {'Train m':<10} {'Val m':<10} {'Val s':<10} {'p95 Thresh':<12} {'Val>p95':}")
print("  " + "-" * 68)
for name, r in results_all.items():
    tr, vr = r["train_patch"], r["val_patch"]
    flagged = (vr > r["thresh_patch"]).sum()
    print(f"  {name:<12} {tr.mean():<10.4f} {vr.mean():<10.4f} {vr.std():<10.4f} "
          f"{r['thresh_patch']:<12.4f} {flagged}/{len(vr)} ({100*flagged/len(vr):.1f}%)")

print(f"\n  Global kNN scoring:")
print(f"  {'Model':<12} {'Train m':<10} {'Val m':<10} {'Val s':<10} {'p95 Thresh':<12} {'Val>p95':}")
print("  " + "-" * 68)
for name, r in results_all.items():
    tr, vr = r["train_knn"], r["val_knn"]
    flagged = (vr > r["thresh_knn"]).sum()
    print(f"  {name:<12} {tr.mean():<10.4f} {vr.mean():<10.4f} {vr.std():<10.4f} "
          f"{r['thresh_knn']:<12.4f} {flagged}/{len(vr)} ({100*flagged/len(vr):.1f}%)")

print(f"\n  Collapse check:")
print(f"  {'Model':<12} {'Feature STD':<14} {'Mean Cos Sim':<14} {'Status'}")
print("  " + "-" * 54)
for name, r in results_all.items():
    feats = r["val_feats"][:2000]
    std_val = np.std(feats, axis=0).mean()
    sim = np.dot(feats, feats.T)
    off = sim[np.triu_indices_from(sim, k=1)].mean()
    status = "Good" if off < 0.9 else "High sim"
    print(f"  {name:<12} {std_val:<14.4f} {off:<14.4f} {status}")

In [ ]:
# Comparative Plots
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("PatchCore — Model Comparison", fontsize=14, y=1.02)

colors = {"Knee": "tab:blue", "Brain": "tab:orange", "Combined": "tab:green"}

for name, r in results_all.items():
    axes[0].hist(r['val_patch'], bins=50, alpha=0.5, label=name, color=colors[name])
axes[0].set_title('Val PatchCore Scores'); axes[0].set_xlabel('Max Patch Dist')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

for name, r in results_all.items():
    axes[1].hist(r['val_knn'], bins=50, alpha=0.5, label=name, color=colors[name])
axes[1].set_title('Val Global kNN Scores'); axes[1].set_xlabel('Cosine Dist')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

for name, r in results_all.items():
    axes[2].scatter(r['val_knn'][:1000], r['val_patch'][:1000],
                    s=3, alpha=0.3, label=name, color=colors[name])
axes[2].set_title('Global kNN vs PatchCore'); axes[2].set_xlabel('kNN')
axes[2].set_ylabel('PatchCore'); axes[2].legend(); axes[2].grid(True, alpha=0.3)

fig.tight_layout()
fig.savefig('patchcore_comparison_plots.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Zip and Download All Checkpoints
import shutil
from IPython.display import FileLink, display

for name, save_dir in [("knee", KNEE_SAVE), ("brain", BRAIN_SAVE), ("combined", COMBINED_SAVE)]:
    zip_path = shutil.make_archive(f"patchcore_{name}_checkpoints", "zip",
                                   root_dir=".", base_dir=save_dir)
    size_mb = os.path.getsize(zip_path) / 1e6
    print(f"  {name}: {zip_path} ({size_mb:.1f} MB)")
    display(FileLink(zip_path))

all_zip = shutil.make_archive("patchcore_all_checkpoints", "zip",
                              root_dir=".", base_dir="/kaggle/working/checkpoints")
print(f"\n  ALL: {all_zip} ({os.path.getsize(all_zip)/1e6:.1f} MB)")
display(FileLink(all_zip))

log_time_budget("SESSION COMPLETE")